In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import pickle

In [ ]:
train = pd.read_csv("train_mushrooms.csv")
test = pd.read_csv("test_mushrooms.csv")
res = pd.read_csv("sample_submission (1).csv")

In [ ]:
data_info = {}

In [ ]:
x = train.drop("class", axis=1)
y = train["class"]

In [ ]:
classes = {"e": 0, "p": 1}

In [ ]:
y = y.map(classes).values
y

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
# x_train = train

In [ ]:

x_train.drop("id", axis=1, inplace=True)

In [ ]:
def prep(df, data_info):
    df = df.copy()

    
    df.drop("id", axis=1, inplace=True)
    df["cap-diameter"] = np.log1p(df["cap-diameter"]).astype("float32")

    df["cap-diameter"] = df["cap-diameter"].fillna(data_info["cap-diameter_log_mean"])
    
    for i in data_info["cap-shape_counts"].keys():
        df[f"cap-shape_{i}"] = (df["cap-shape"] == i).astype("int8")
    df.drop("cap-shape", axis=1, inplace=True)
    
    df["cap-surface"] = df["cap-surface"].apply(lambda x: x if x in data_info["cap-surface_counts"].keys() else np.random.choice(data_info["cap-surface_counts"].keys(), p=data_info["cap-surface_counts"].values))
    for i in data_info["cap-surface_counts"].keys():
        df[f"cap-surface_{i}"] = (df["cap-surface"] == i).astype("int8")
    df.drop("cap-surface", axis=1, inplace=True)
    
    for i in data_info["cap-color_counts"].keys():
        df[f"cap-color_{i}"] = (df["cap-color"] == i).astype("int8")
    df.drop("cap-color", axis=1, inplace=True)
    
    df["does-bruise-or-bleed"] = df["does-bruise-or-bleed"].apply(lambda x: 1 if x == "t" else 0).astype("int8")
    
    df["gill-attachment"] = df["gill-attachment"].apply(
        lambda x: x if x in data_info["gill-attachment_counts"].keys() \
                  else np.random.choice(data_info["gill-attachment_counts"].keys(), p=data_info["gill-attachment_counts"].values))
    for i in data_info["gill-attachment_counts"].keys():
        df[f"gill-attachment_{i}"] = (df["gill-attachment"] == i).astype("int8")
    df.drop("gill-attachment", axis=1, inplace=True)
    df.drop(["gill-spacing", "stem-root", "stem-surface", "veil-type", "veil-color", "spore-print-color"], axis=1, inplace=True)
    
    for i in data_info["gill-color_counts"].keys():
        df[f"gill-color_{i}"] = (df["gill-color"] == i).astype("int8")
    df.drop("gill-color", axis=1, inplace=True)
    df["stem-height"] = np.log1p(df["stem-height"]).astype("float32")
    
    df["stem-height"] = df["stem-height"].fillna(data_info["stem-height_log_mean"])
    df["stem-width"] = np.log1p(df["stem-width"]).astype("float32")
    
    df["stem-width"] = df["stem-width"].fillna(data_info["stem-width_log_mean"])
    
    for i in data_info["stem-color_counts"].keys():
        df[f"stem-color_{i}"] = (df["stem-color"] == i).astype("int8")
    df.drop("stem-color", axis=1, inplace=True)
    df["has-ring"] = df["has-ring"].apply(lambda x: 1 if x=='t' else 0).astype("int8")
    
    df["ring-type"] = df["ring-type"].apply(
        lambda x: x if x in data_info["ring-type_counts"].keys() \
                  else np.random.choice(data_info["ring-type_counts"].keys(), p=data_info["ring-type_counts"].values))
    for i in data_info["ring-type_counts"].keys():
        df[f"ring-type_{i}"] = (df["ring-type"] == i).astype("int8")
    df.drop("ring-type", axis=1, inplace=True)
    
    for i in data_info["habitat_counts"].keys():
        df[f"habitat_{i}"] = (df["habitat"] == i).astype("int8")
    df.drop("habitat", axis=1, inplace=True)
    
    for i in data_info["season_counts"].keys():
        df[f"season_{i}"] = (df["season"] == i).astype("int8")
    df.drop("season", axis=1, inplace=True)
    num_columns = ["cap-diameter", "stem-height","stem-width"]
    
    df[num_columns] = data_info["sc"].transform(df[num_columns])

    return df

In [ ]:
x_train["cap-diameter"] = np.log1p(x_train["cap-diameter"]).astype("float32")
data_info["cap-diameter_log_mean"] = x_train["cap-diameter"].mean()
x_train["cap-diameter"] = x_train["cap-diameter"].fillna(data_info["cap-diameter_log_mean"])

In [ ]:
data_info["cap-shape_counts"] = x_train["cap-shape"].value_counts()[:7] / x_train["cap-shape"].value_counts()[:7].sum()
for i in data_info["cap-shape_counts"].keys():
    x_train[f"cap-shape_{i}"] = (x_train["cap-shape"] == i).astype("int8")
x_train.drop("cap-shape", axis=1, inplace=True)

In [ ]:
data_info["cap-surface_counts"] = x_train["cap-surface"].value_counts()[:11] / x_train["cap-surface"].value_counts()[:11].sum()
x_train["cap-surface"] = x_train["cap-surface"].apply(lambda x: x if x in data_info["cap-surface_counts"].keys() else np.random.choice(data_info["cap-surface_counts"].keys(), p=data_info["cap-surface_counts"].values))
for i in data_info["cap-surface_counts"].keys():
    x_train[f"cap-surface_{i}"] = (x_train["cap-surface"] == i).astype("int8")
x_train.drop("cap-surface", axis=1, inplace=True)

In [ ]:
data_info["cap-color_counts"] = x_train["cap-color"].value_counts()[:12] / x_train["cap-color"].value_counts()[:12].sum()
for i in data_info["cap-color_counts"].keys():
    x_train[f"cap-color_{i}"] = (x_train["cap-color"] == i).astype("int8")
x_train.drop("cap-color", axis=1, inplace=True)

In [ ]:
data_info["cap-bb_counts"] = x_train["does-bruise-or-bleed"].value_counts()[:2] / x_train["does-bruise-or-bleed"].value_counts()[:2].sum()
x_train["does-bruise-or-bleed"] = x_train["does-bruise-or-bleed"].apply(lambda x: 1 if x == "t" else 0).astype("int8")

In [ ]:
data_info["gill-attachment_counts"] = x_train["gill-attachment"].value_counts()[:7] / x_train["gill-attachment"].value_counts()[:7].sum()
x_train["gill-attachment"] = x_train["gill-attachment"].apply(
    lambda x: x if x in data_info["gill-attachment_counts"].keys() \
              else np.random.choice(data_info["gill-attachment_counts"].keys(), p=data_info["gill-attachment_counts"].values))
for i in data_info["gill-attachment_counts"].keys():
    x_train[f"gill-attachment_{i}"] = (x_train["gill-attachment"] == i).astype("int8")
x_train.drop("gill-attachment", axis=1, inplace=True)

In [ ]:
x_train.drop(["gill-spacing", "stem-root", "stem-surface", "veil-type", "veil-color", "spore-print-color"], axis=1, inplace=True)

In [ ]:
data_info["gill-color_counts"] = x_train["gill-color"].value_counts()[:12] / x_train["gill-color"].value_counts()[:12].sum()
for i in data_info["gill-color_counts"].keys():
    x_train[f"gill-color_{i}"] = (x_train["gill-color"] == i).astype("int8")
x_train.drop("gill-color", axis=1, inplace=True)

In [ ]:
x_train["stem-height"] = np.log1p(x_train["stem-height"]).astype("float32")
data_info["stem-height_log_mean"] = x_train["stem-height"].mean()
x_train["stem-height"] = x_train["stem-height"].fillna(data_info["stem-height_log_mean"])

In [ ]:
x_train["stem-width"] = np.log1p(x_train["stem-width"]).astype("float32")
data_info["stem-width_log_mean"] = x_train["stem-width"].mean()
x_train["stem-width"] = x_train["stem-width"].fillna(data_info["stem-width_log_mean"])

In [ ]:
data_info["stem-color_counts"] = x_train["stem-color"].value_counts()[:12] / x_train["stem-color"].value_counts()[:12].sum()
for i in data_info["stem-color_counts"].keys():
    x_train[f"stem-color_{i}"] = (x_train["stem-color"] == i).astype("int8")
x_train.drop("stem-color", axis=1, inplace=True)

In [ ]:
x_train["has-ring"] = x_train["has-ring"].apply(lambda x: 1 if x=='t' else 0).astype("int8")

In [ ]:
data_info["ring-type_counts"] = x_train["ring-type"].value_counts()[:7] / x_train["ring-type"].value_counts()[:7].sum()
x_train["ring-type"] = x_train["ring-type"].apply(
    lambda x: x if x in data_info["ring-type_counts"].keys() \
              else np.random.choice(data_info["ring-type_counts"].keys(), p=data_info["ring-type_counts"].values))
for i in data_info["ring-type_counts"].keys():
    x_train[f"ring-type_{i}"] = (x_train["ring-type"] == i).astype("int8")
x_train.drop("ring-type", axis=1, inplace=True)

In [ ]:
data_info["habitat_counts"] = x_train["habitat"].value_counts()[:7] / x_train["habitat"].value_counts()[:7].sum()
for i in data_info["habitat_counts"].keys():
    x_train[f"habitat_{i}"] = (x_train["habitat"] == i).astype("int8")
x_train.drop("habitat", axis=1, inplace=True)

In [ ]:
data_info["season_counts"] = x_train["season"].value_counts()[:4] / x_train["season"].value_counts()[:4].sum()
for i in data_info["season_counts"].keys():
    x_train[f"season_{i}"] = (x_train["season"] == i).astype("int8")
x_train.drop("season", axis=1, inplace=True)

In [ ]:
num_columns = ["cap-diameter", "stem-height","stem-width"]

In [ ]:
sc = StandardScaler()
x_train[num_columns] = sc.fit_transform(x_train[num_columns])

In [ ]:
data_info["sc"] = sc

In [ ]:
test = prep(test, data_info)

In [ ]:
x_test = prep(x_test, data_info)

In [ ]:
x = train.drop("class", axis=1)
y = train["class"]
y = y.map(classes).values

In [ ]:
x_train

In [ ]:
x_test

In [ ]:
# def predict(model):
#     model.fit(x_train, y_train)
#     train_pred = model.predict(x_train)
#     test_pred = model.predict(x_test)
#     print(f'Train predict = {accuracy_score(y_train, train_pred)}')
#     print(f'Test predict = {accuracy_score(y_test, test_pred)}')
#     return model

def predict(model):
    model.fit(x, y)
    train_pred = model.predict(x)
    test_pred = model.predict(test)
    print(f'Train predict = {accuracy_score(y, train_pred)}')
    return model, test_pred
    

In [ ]:
model = tf.keras.Sequential([
    Input(x_train.shape[1:], 32),
    Flatten(),
    Dense(1000, activation='sigmoid'),
    Dense(1000, activation='relu'),
    Dense(10, activation='softmax')
])
model.summary()

In [ ]:
model.compile(optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']) #0.8748, 

In [ ]:
model.fit(x_train, y_train, 32, 5, validation_data=(x_test, y_test))

In [ ]:
preds = model.predict(x_test).argmax(axis=1)

In [ ]:
model, preds = predict(DecisionTreeClassifier(max_depth=30, min_samples_split=5, min_samples_leaf=5, random_state=42))

In [ ]:
data_info["model"] = model

In [ ]:
with open("mushrooms_info.pck", "wb") as f:
    pickle.dump(data_info, f)

In [ ]:
classes_int2str = {j: i for i, j in classes.items()}

In [ ]:
classes_int2str

In [ ]:
p = np.array(list(map(lambda x: "e" if x==0 else "p", preds)))

In [ ]:
p

In [ ]:
res["class"] = p

In [ ]:
res

In [ ]:
res.to_csv("mushroom_score.csv", index=False)